In [ ]:
!pip install transformers datasets accelerate evaluate rouge_score

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline
)
from datasets import Dataset
import evaluate

# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Define Clinical Note Templates

### Subtask:
Create data structures and templates for generating synthetic H&P, Progress Notes, and Discharge Summaries.


In [ ]:
import random
from datetime import datetime, timedelta

# 1. Define Clinical Knowledge Base
clinical_knowledge_base = {
    'Pneumonia': {
        'symptoms': ['fever', 'cough with sputum', 'shortness of breath', 'chest pain'],
        'medications': ['Azithromycin', 'Ceftriaxone', 'Acetaminophen'],
        'labs': ['WBC 15,000', 'C-Reactive Protein elevated'],
        'imaging': 'CXR shows right lower lobe consolidation',
        'plan': 'Start IV antibiotics, monitor O2 saturation, sputum culture.'
    },
    'Heart Failure': {
        'symptoms': ['shortness of breath', 'orthopnea', 'leg swelling', 'fatigue'],
        'medications': ['Furosemide', 'Lisinopril', 'Carvedilol'],
        'labs': ['BNP 1200', 'Creatinine 1.4'],
        'imaging': 'ECHO shows EF 35%',
        'plan': 'Diuresis with Lasix, monitor electrolytes, daily weights.'
    },
    'Cellulitis': {
        'symptoms': ['redness', 'swelling', 'warmth', 'pain in lower leg'],
        'medications': ['Cephalexin', 'Clindamycin', 'Ibuprofen'],
        'labs': ['WBC 11,000', 'ESR elevated'],
        'imaging': 'Ultrasound negative for DVT',
        'plan': 'Antibiotic therapy, elevation of limb, mark margins.'
    }
}

# 2. Function to generate H&P Template
def generate_hp_template(patient_data):
    condition = patient_data['condition']
    kb = clinical_knowledge_base[condition]

    hp_text = f"""
HISTORY OF PRESENT ILLNESS:
Patient is a {patient_data['age']} year old {patient_data['gender']} presenting with {', '.join(kb['symptoms'])}.
Symptoms started {random.randint(2, 7)} days ago and have been worsening.

PAST MEDICAL HISTORY:
{patient_data.get('pmh', 'Hypertension, Hyperlipidemia')}

MEDICATIONS:
{', '.join(kb['medications'])}

ALLERGIES:
NKDA

SOCIAL HISTORY:
{random.choice(['Does not smoke', 'Smokes 1ppd', 'Former smoker'])}

REVIEW OF SYSTEMS:
Positive for constitutional symptoms as noted above. All other systems negative.

PHYSICAL EXAM:
Vitals: T {random.randint(98, 102)}, HR {random.randint(60, 110)}, BP {random.randint(100, 150)}/{random.randint(60, 90)}
General: Alert and oriented.
Lungs/Heart: See findings below.
"""
    return hp_text

# 3. Function to generate Progress Note Template
def generate_progress_note_template(patient_data, day):
    condition = patient_data['condition']
    kb = clinical_knowledge_base[condition]

    progress_note = f"""
PROGRESS NOTE - Day {day}

SUBJECTIVE:
Patient reports feeling {random.choice(['better', 'about the same', 'slightly worse'])} today.
{random.choice(kb['symptoms']).capitalize()} is {random.choice(['improving', 'stable', 'persisting'])}.

OBJECTIVE:
Vitals: Stable.
Labs: {', '.join(kb['labs'])}
Imaging: {kb['imaging']}

ASSESSMENT:
{condition}, currently {random.choice(['improving', 'stable'])}.

PLAN:
Continue current management.
{kb['plan']}
"""
    return progress_note

# 4. Function to generate Discharge Summary Template
def generate_discharge_summary_template(patient_data):
    condition = patient_data['condition']
    kb = clinical_knowledge_base[condition]

    discharge_summary = f"""
DISCHARGE SUMMARY

ADMISSION DIAGNOSIS:
{condition}

HOSPITAL COURSE:
Patient was admitted with {', '.join(kb['symptoms'])}. Treated with {kb['medications'][0]}.
Condition improved over hospital stay. Vitals stabilized.

DISCHARGE MEDICATIONS:
{', '.join(kb['medications'])}

FOLLOW-UP:
Follow up with PCP in 1 week.
Return to ED if symptoms worsen.
"""
    return discharge_summary

# 5. Helper function for Gold Standard Summary
def generate_gold_standard_summary(note_type, patient_data):
    condition = patient_data['condition']
    kb = clinical_knowledge_base[condition]

    if note_type == 'H&P':
        return f"{patient_data['age']}yo {patient_data['gender']} presents with {condition} symptoms including {kb['symptoms'][0]} and {kb['symptoms'][1]}. History consistent with acute {condition}."
    elif note_type == 'Progress Note':
        return f"Day {patient_data.get('day', '?')} update for {condition}: Patient is stable/improving. Labs show {kb['labs'][0]}. Plan: Continue {kb['medications'][0]}."
    elif note_type == 'Discharge Summary':
        return f"Patient admitted for {condition}, treated with {kb['medications'][0]} and improved. Discharged home with follow-up in 1 week."
    return "Summary not available."

## Generate Synthetic Patient Encounters

### Subtask:
Generate a dataset of clinical notes (H&P, Progress Notes, Discharge Summaries) for multiple synthetic patients using the defined templates.


In [ ]:
dataset = []

for i in range(10000):
    # Randomly assign patient demographics and condition
    condition = random.choice(list(clinical_knowledge_base.keys()))
    age = random.randint(18, 90)
    gender = random.choice(['Male', 'Female'])

    patient_data = {
        'condition': condition,
        'age': age,
        'gender': gender
    }

    # Generate H&P
    hp_text = generate_hp_template(patient_data)
    hp_summary = generate_gold_standard_summary('H&P', patient_data)
    dataset.append({
        'input_text': hp_text,
        'target_text': hp_summary,
        'note_type': 'H&P'
    })

    # Generate Progress Notes (Simulate 3 days)
    for day in range(1, 4):
        patient_data['day'] = day
        progress_note = generate_progress_note_template(patient_data, day)
        progress_summary = generate_gold_standard_summary('Progress Note', patient_data)
        dataset.append({
            'input_text': progress_note,
            'target_text': progress_summary,
            'note_type': 'Progress Note'
        })

    # Generate Discharge Summary
    discharge_summary = generate_discharge_summary_template(patient_data)
    discharge_summary_gold = generate_gold_standard_summary('Discharge Summary', patient_data)
    dataset.append({
        'input_text': discharge_summary,
        'target_text': discharge_summary_gold,
        'note_type': 'Discharge Summary'
    })

# Convert to DataFrame
df_clinical_notes = pd.DataFrame(dataset)

# Display data info
print(f"Dataset Shape: {df_clinical_notes.shape}")
df_clinical_notes.head()

## Save and Review Dataset

### Subtask:
Save the generated clinical notes dataset to a CSV file and review a few examples to ensure data quality.


In [ ]:
# Save to CSV
df_clinical_notes.to_csv('clinical_notes_dataset.csv', index=False)
print("Dataset saved to 'clinical_notes_dataset.csv'")

# Display first few rows
print(df_clinical_notes.head())

# Inspect one example of each type
print("\n--- Data Inspection ---")
for note_type in ['H&P', 'Progress Note', 'Discharge Summary']:
    example = df_clinical_notes[df_clinical_notes['note_type'] == note_type].iloc[0]
    print(f"\n=== {note_type} ===")
    print("INPUT TEXT:")
    print(example['input_text'])
    print("TARGET SUMMARY:")
    print(example['target_text'])

## Data Preparation for Modeling

### Subtask:
Prepare the dataset for the Seq2Seq model by converting it to a Hugging Face Dataset, splitting it into train/test sets, and tokenizing the text.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_clinical_notes)

# 2. Split into Train and Test
dataset_split = hf_dataset.train_test_split(test_size=0.2, seed=42)

# 3. Initialize Tokenizer
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 4. Define Preprocessing Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True, padding="max_length")

    # Fix: as_target_tokenizer is deprecated. Use text_target argument instead.
    labels = tokenizer(text_target=examples["target_text"], max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 5. Apply Preprocessing
tokenized_datasets = dataset_split.map(preprocess_function, batched=True)

# 6. Check Structure
print(tokenized_datasets)

## Train Summarization Model

### Subtask:
Fine-tune a T5-small model on the clinical notes dataset using the Seq2SeqTrainer.


In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. Instantiate Model
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# 2. Define Training Arguments
batch_size = 8
args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=(device == "cuda"),  # Use mixed precision if on GPU
)

# 3. Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 4. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer, # Changed from tokenizer to processing_class
    data_collator=data_collator,
)

# 5. Train
trainer.train()

**Reasoning**:
The training has successfully completed. Now I need to evaluate the model on the test set to check its performance using ROUGE scores.



In [ ]:
import numpy as np
import evaluate

# Load ROUGE metric
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# Define trainer with compute_metrics for evaluation
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Run evaluation
eval_results = trainer.evaluate()
print(eval_results)

## Evaluate and Test

### Subtask:
Generate summaries for a few test cases using the fine-tuned model and visually compare them with the gold standard summaries.


In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, AutoTokenizer
from datasets import Dataset
import torch
import random
import numpy as np

# --- 1. Fix Data Preprocessing and Re-train ---

# Re-load tokenizer and model to start fresh
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Correct Preprocessing: Do NOT pad labels here. Let DataCollator handle padding and -100 masking.
def preprocess_function_correct(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True)

    # Tokenize targets but don't pad yet
    labels = tokenizer(text_target=examples["target_text"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Re-process dataset
hf_dataset = Dataset.from_pandas(df_clinical_notes)
dataset_split = hf_dataset.train_test_split(test_size=0.2, seed=42)
tokenized_datasets_fixed = dataset_split.map(preprocess_function_correct, batched=True)

# Data Collator (handles padding and label masking)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

# Training Arguments
args = Seq2SeqTrainingArguments(
    output_dir="./results_fixed",
    eval_strategy="epoch",
    learning_rate=2e-4, # Slightly higher LR for quick convergence on synthetic data
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5, # Increase epochs slightly
    predict_with_generate=True,
    fp16=(device == "cuda"),
    logging_steps=50,
    report_to="none"
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets_fixed["train"],
    eval_dataset=tokenized_datasets_fixed["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Re-training model with corrected data processing...")
trainer.train()

# --- 2. Generate Summaries with Fixed Model ---

print("\n--- Model Evaluation on Test Cases (Fixed Model) ---")
test_dataset = tokenized_datasets_fixed["test"]
random_indices = random.sample(range(len(test_dataset)), 5)

for idx in random_indices:
    example = test_dataset[idx]
    input_text = example["input_text"]
    gold_summary = example["target_text"]
    note_type = example["note_type"]

    # Prepare input
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=128,
        min_length=10,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

    generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print(f"\n=== Example {idx} ({note_type}) ===")
    print(f"INPUT TEXT (Truncated): {input_text[:150]}...")
    print(f"GOLD STANDARD: {gold_summary}")
    print(f"MODEL GENERATED: {generated_summary}")

## Final Task

### Subtask:
Summarize the dataset creation process and analyze the fine-tuned model's performance across different clinical note types.


## Summary:

### Data Analysis Key Findings

*   **Dataset Generation**: A synthetic dataset of **500 clinical notes** was successfully generated, representing **100 unique patient encounters**. Each encounter included one History of Present Illness (H&P), three Progress Notes, and one Discharge Summary.
*   **Clinical Scope**: The dataset covered three distinct medical conditions: **Pneumonia, Heart Failure, and Cellulitis**, with structured templates ensuring consistent inclusion of symptoms, medications, labs, and imaging data.
*   **Model Training Configuration**: The dataset was split into **400 training examples** and **100 test examples**. A **T5-small** model was fine-tuned using a Seq2Seq architecture.
*   **Technical Challenges and Resolution**:
    *   Initial training resulted in **0.0 ROUGE scores** and failed generation because the target labels were padded incorrectly (using the pad token ID instead of masking with -100).
    *   Retraining with a corrected data collator, increased epochs (5), and an adjusted learning rate (2e-4) resolved the issue.
*   **Performance Outcome**: Post-correction qualitative evaluation confirmed that the model could successfully generate coherent summaries. For example, it accurately condensed a lengthy H&P note into a concise summary stating the patient's age, gender, presenting symptoms, and history consistent with the specific diagnosis.

### Insights or Next Steps

*   **Quantitative Evaluation**: While visual inspection confirms the model works, the next logical step is to run ROUGE and BLEU metric calculations on the *fixed* model to obtain a numerical baseline for performance.
*   **Knowledge Base Expansion**: The current model is limited to three conditions. To improve generalization, the synthetic generation script should be expanded to include a wider variety of diagnoses, negations (e.g., "patient denies chest pain"), and more complex clinical narratives.
